In [9]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from BaselineRemoval import BaselineRemoval
import matplotlib.pyplot as plt

data = pd.read_csv('M0.csv')

n =1300
data_separate = pd.DataFrame(np.hstack(np.vsplit(data.values, len(data) // n)))
data_separate.columns = data_separate.columns.map(lambda c: chr(c + ord('A')))



def getDuplicateColumns(df):
  
    # Create an empty set
    duplicateColumnNames = set()
      
    # Iterate through all the columns 
    # of dataframe
    for x in range(data_separate.shape[1]):
          
        # Take column at xth index.
        col = data_separate.iloc[:, x]
          
        # Iterate through all the columns in
        # DataFrame from (x + 1)th index to
        # last index
        for y in range(x + 1, data_separate.shape[1]):
              
            # Take column at yth index.
            otherCol = data_separate.iloc[:, y]
              
            # Check if two columns at x & y
            # index are equal or not,
            # if equal then adding 
            # to the set
            if col.equals(otherCol):
                duplicateColumnNames.add(data_separate.columns.values[y])
                  
    # Return list of unique column names 
    # whose contents are duplicates.
    return list(duplicateColumnNames)

rslt_df = data_separate.drop(columns = getDuplicateColumns(data_separate))
    
rslt_df.rename(columns = {'A':'Wavenumber'}, inplace = True)

#rslt_df.to_excel('M1 column separation.xlsx', index=False)

In [10]:
#Cosmic ray removal    
#function
def modified_z_score(intensity):
 median_int = np.median(intensity)
 mad_int = np.median([np.abs(intensity - median_int)])
 modified_z_scores = 0.6745 * (intensity - median_int) / mad_int
 return modified_z_scores


def fixer(y,m):
 threshold = 5 # binarization threshold. 
 spikes = abs(np.array(modified_z_score(np.diff(y)))) > threshold
 y_out = y.copy() # So we don’t overwrite y
 for i in np.arange(len(spikes)):
  if spikes[i] != 0: # If we have an spike in position i
   w = np.arange(i-m,i+1+m) # we select 2 m + 1 points around our spike
   w2 = w[spikes[w] == 0] # From such interval, we choose the ones which are not spikes
   y_out[i] = np.mean(y[w2])  # and we average their values
 return y_out

# Transform the data to a numpy array
#data1 = rslt_df.drop(columns = ['Wavenumber'])
MASTER = []
for i in rslt_df.columns:
    intensity = rslt_df[i].values.tolist()
    intensity = np.array(intensity)
    fixed_intensity = fixer(intensity,m=7)  
    MASTER.append(fixed_intensity)
    
df = pd.DataFrame(MASTER)
result = df.transpose()
result.columns = result.columns.map(lambda c: chr(c + ord('A')))
#result.to_excel('M1 cosmic ray removal.xlsx', index=False)


In [11]:
#baseline removal 

data1 = result.drop(columns = ['A'])

cols = list(data1.columns.values)
#ind = data. iloc[:, 0]
#ind_list = ind.tolist()

Master2=[]

for i in data1.columns:
    temp = data1[i].values.tolist()
    baseObj=BaselineRemoval(temp)
    Zhangfit_output=baseObj.ZhangFit()
    Master2.append(Zhangfit_output)
    
output = pd.DataFrame (Master2)

reshape = np.transpose(output)

#reshape.to_excel('M1 baseline correction.xlsx', index=False)


In [12]:
#Normalization

from sklearn import preprocessing

x = reshape.values #returns a numpy array
min_max_scaler = preprocessing.MinMaxScaler()
x_scaled = min_max_scaler.fit_transform(x)
data_normalized = pd.DataFrame(x_scaled, columns = [cols])

#output
new_df = rslt_df['Wavenumber']
a = new_df.to_frame(name='Wavenumber')

frame = pd.concat([a,data_normalized], axis = 1)
frame.to_excel('M0 out.xlsx', index=False)
